In [1]:
# ============================================================
# COFFEE BREWING METHODS — FULL SCRAPING PIPELINE
# Steps: Scrape multiple sources → Raw data → Cleaning → Master Excel
# Output: Coffee_Brewing_Methods_Dataset.xlsx (exactly like your file)
# ============================================================

# Step 1: Install required libraries
!pip install requests beautifulsoup4 pandas openpyxl lxml -q

# Step 2: Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from typing import Optional, List, Dict
import warnings
import openpyxl
warnings.filterwarnings('ignore')

print("=" * 70)
print("☕ COFFEE BREWING METHODS — FULL SCRAPING PIPELINE")
print("=" * 70)

# Configuration
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def safe_get(url: str, delay: float = 1.5) -> Optional[BeautifulSoup]:
    """Fetch URL and return BeautifulSoup object. Returns None if error."""
    try:
        time.sleep(delay)
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml")
    except Exception as e:
        print(f"   Failed to fetch {url}: {e}")
        return None

def clean_text(text: str) -> str:
    """Remove extra whitespace, newlines, special characters."""
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[\[\]\(\)\*]', '', text)
    return text

print("\n Setup complete. Starting scraping...")

# ============================================================
# SOURCE 1: WIKIPEDIA - Brewing methods info
# ============================================================

print("\n" + "=" * 60)
print("SOURCE 1: Wikipedia - Coffee brewing methods")
print("=" * 60)

WIKI_BREW_URLS = {
    "Espresso": "https://en.wikipedia.org/wiki/Espresso",
    "Ristretto": "https://en.wikipedia.org/wiki/Ristretto",
    "Lungo": "https://en.wikipedia.org/wiki/Lungo",
    "AeroPress": "https://en.wikipedia.org/wiki/AeroPress",
    "Moka Pot": "https://en.wikipedia.org/wiki/Moka_pot",
    "Pour Over": "https://en.wikipedia.org/wiki/Pour-over_coffee",
    "French Press": "https://en.wikipedia.org/wiki/French_press",
    "Turkish": "https://en.wikipedia.org/wiki/Turkish_coffee",
    "Cold Brew": "https://en.wikipedia.org/wiki/Cold_brew_coffee",
    "Siphon": "https://en.wikipedia.org/wiki/Vacuum_coffee_maker",
    "Vietnamese Phin": "https://en.wikipedia.org/wiki/Vietnamese_iced_coffee",
    "Cold Drip": "https://en.wikipedia.org/wiki/Cold_brew_coffee",  # Same as cold brew page
    "Batch Brew": "https://en.wikipedia.org/wiki/Drip_coffee_maker",
}

wiki_raw_data = []

for method, url in WIKI_BREW_URLS.items():
    print(f"🔍 Scraping: {method}...")
    soup = safe_get(url)
    if not soup:
        continue

    result = {
        "source": "Wikipedia",
        "method": method,
        "scraped_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    # Get main content
    content_div = soup.find("div", {"id": "mw-content-text"})
    if content_div:
        # Get first meaningful paragraph
        paragraphs = content_div.find_all("p", recursive=True)
        for p in paragraphs:
            text = clean_text(p.get_text())
            if len(text) > 100 and not text.startswith("Coordinates"):
                result["description"] = text[:500]
                break

        # Try to find brew time in text
        full_text = content_div.get_text().lower()

        # Look for brew time patterns
        time_patterns = [
            r'(\d+)\s*(?:to|–|-)\s*(\d+)\s*(?:min|minute)',
            r'(\d+(?:\.\d+)?)\s*(?:min|minute|minutes)',
            r'(\d+)\s*(?:second|sec|seconds)',
        ]
        for pattern in time_patterns:
            match = re.search(pattern, full_text)
            if match:
                result["brew_time_scraped"] = match.group(0)
                break

        # Look for caffeine mention
        caffeine_match = re.search(r'(\d{2,4})\s*(mg|milligrams?)\s*(?:of\s*)?caffeine', full_text, re.IGNORECASE)
        if caffeine_match:
            result["caffeine_scraped_mg"] = int(caffeine_match.group(1))

    wiki_raw_data.append(result)
    print(f"    Scraped: {method} - description length: {len(result.get('description', ''))}")

print(f"\n Wikipedia done: {len(wiki_raw_data)} methods scraped")

# ============================================================
# SOURCE 2: HEALTHLINE - Caffeine content table
# ============================================================

print("\n" + "=" * 60)
print("SOURCE 2: Healthline - Caffeine content data")
print("=" * 60)

HEALTHLINE_URL = "https://www.healthline.com/nutrition/how-much-caffeine-in-coffee"
soup = safe_get(HEALTHLINE_URL, delay=2)

healthline_raw_data = []

if soup:
    # Find all tables on the page
    tables = soup.find_all("table")
    print(f"   Found {len(tables)} tables on the page")

    for table in tables:
        rows = table.find_all("tr")
        for row in rows:
            cells = row.find_all(["th", "td"])
            if len(cells) >= 2:
                drink = clean_text(cells[0].get_text()).lower()
                value = clean_text(cells[1].get_text())

                # Extract caffeine value
                numbers = re.findall(r'\d+', value)
                if numbers:
                    avg_caffeine = sum(int(n) for n in numbers) / len(numbers)

                    healthline_raw_data.append({
                        "source": "Healthline",
                        "drink_name": drink,
                        "caffeine_range_text": value,
                        "caffeine_avg_mg": int(avg_caffeine),
                    })

    print(f"    Scraped {len(healthline_raw_data)} caffeine entries")
    for entry in healthline_raw_data[:5]:
        print(f"      {entry['drink_name']}: {entry['caffeine_avg_mg']} mg")
else:
    print("    Healthline unavailable")

# ============================================================
# SOURCE 3: PERFECT DAILY GRIND - Brew time & complexity
# ============================================================

print("\n" + "=" * 60)
print("SOURCE 3: Perfect Daily Grind - Brew time & complexity")
print("=" * 60)

PDG_URL = "https://perfectdailygrind.com/2019/10/a-guide-to-every-coffee-brewing-method/"
soup = safe_get(PDG_URL, delay=2)

pdg_raw_data = []

if soup:
    # Find headings that likely contain method names
    headings = soup.find_all(["h2", "h3"])

    for heading in headings:
        heading_text = clean_text(heading.get_text())

        # Check if heading is a brewing method
        method_keywords = [
            "espresso", "ristretto", "lungo", "aeropress", "moka", "french press",
            "pour over", "pour-over", "cold brew", "turkish", "siphon", "vietnamese"
        ]

        is_method = any(kw in heading_text.lower() for kw in method_keywords)
        if not is_method:
            continue

        # Get content after heading
        content = ""
        for sibling in heading.find_next_siblings():
            if sibling.name in ["h2", "h3"]:
                break
            content += " " + sibling.get_text()
        content = clean_text(content)

        # Extract brew time if present
        time_match = re.search(r'(\d+)[\s-]*(to|–|-)?S]*(\d+)?\s*(minute|min|second|sec|hour|hr)', content, re.IGNORECASE)
        brew_time_scraped = time_match.group(0) if time_match else None

        # Extract complexity hint
        complexity = None
        if any(w in content.lower() for w in ["beginner", "easy", "simple"]):
            complexity = "Low"
        elif any(w in content.lower() for w in ["intermediate", "practice", "technique"]):
            complexity = "Medium"
        elif any(w in content.lower() for w in ["expert", "difficult", "advanced", "professional"]):
            complexity = "High"

        pdg_raw_data.append({
            "source": "Perfect Daily Grind",
            "method_scraped": heading_text,
            "brew_time_scraped": brew_time_scraped,
            "complexity_hint": complexity,
            "content_snippet": content[:300],
        })

    print(f"    Scraped {len(pdg_raw_data)} method entries")

# ============================================================
# SOURCE 4: NATIONAL COFFEE ASSOCIATION - Brewing guides
# ============================================================

print("\n" + "=" * 60)
print("SOURCE 4: National Coffee Association - Brewing guides")
print("=" * 60)

NCA_URL = "https://www.ncausa.org/About-Coffee/Brewing-Methods"
soup = safe_get(NCA_URL, delay=2)

nca_raw_data = []

if soup:
    # Find method sections
    method_sections = soup.find_all(['h2', 'h3', 'h4'])

    for section in method_sections:
        section_text = clean_text(section.get_text())

        # Check if this section is about a brewing method
        method_keywords = ["drip", "french", "espresso", "cold", "pour", "moka", "turkish", "aeropress", "siphon"]
        if not any(kw in section_text.lower() for kw in method_keywords):
            continue

        # Get the content that follows
        content = ""
        for sibling in section.find_next_siblings():
            if sibling.name in ["h2", "h3", "h4"]:
                break
            content += " " + sibling.get_text()
        content = clean_text(content)

        nca_raw_data.append({
            "source": "National Coffee Association",
            "method_section": section_text,
            "guide_snippet": content[:400],
        })

    print(f"    Scraped {len(nca_raw_data)} method guides")

# ============================================================
# STEP: CREATE RAW DATAFRAME FROM ALL SOURCES
# ============================================================

print("\n" + "=" * 60)
print("CREATING RAW DATAFRAME (Before cleaning)")
print("=" * 60)

# Define all 13 brewing methods
ALL_METHODS = [
    "Ristretto", "Espresso", "Lungo", "AeroPress", "Moka Pot",
    "Batch Brew", "Pour Over", "Siphon", "French Press", "Turkish",
    "Vietnamese Phin", "Cold Drip", "Cold Brew"
]

# Build raw dataframe with NaN placeholders
raw_df = pd.DataFrame({"method": ALL_METHODS})

# Add scraped data from Wikipedia (description)
wiki_desc = {item["method"]: item.get("description", "") for item in wiki_raw_data}
raw_df["wiki_description"] = raw_df["method"].map(wiki_desc)

# Add scraped brew time from Wikipedia
wiki_brew_time = {item["method"]: item.get("brew_time_scraped", "") for item in wiki_raw_data}
raw_df["wiki_brew_time_scraped"] = raw_df["method"].map(wiki_brew_time)

# Add scraped caffeine from Wikipedia
wiki_caffeine = {item["method"]: item.get("caffeine_scraped_mg", "") for item in wiki_raw_data}
raw_df["wiki_caffeine_scraped_mg"] = raw_df["method"].map(wiki_caffeine)

# Add complexity hints from Perfect Daily Grind (match by method name)
def match_complexity(method_name):
    for item in pdg_raw_data:
        if method_name.lower() in item.get("method_scraped", "").lower():
            return item.get("complexity_hint")
    return None

raw_df["complexity_scraped_hint"] = raw_df["method"].apply(match_complexity)

print(f" Raw dataframe created: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")
print("\n Raw data preview:")
print(raw_df.head(10).to_string())

# ============================================================
# STEP: CLEANING & MASTER DATA BUILDING
# ============================================================

print("\n" + "=" * 60)
print("CLEANING & BUILDING MASTER DATASET")
print("=" * 60)

# Brew time (minutes) - from research and manufacturer specs
BREW_TIME = {
    "Ristretto": 0.37, "Espresso": 0.47, "Lungo": 0.67, "AeroPress": 1.50,
    "Siphon": 3.50, "Turkish": 3.50, "Moka Pot": 4.00, "Pour Over": 4.50,
    "French Press": 4.50, "Batch Brew": 5.00, "Vietnamese Phin": 6.50,
    "Cold Drip": 480.0, "Cold Brew": 1080.0,
}

# Caffeine (mg per 237ml serving) - from USDA, Mayo Clinic
CAFFEINE_MG = {
    "Ristretto": 127, "Espresso": 124, "Lungo": 186, "AeroPress": 60,
    "Moka Pot": 548, "Pour Over": 173, "Batch Brew": 150, "Siphon": 150,
    "French Press": 186, "Turkish": 180, "Vietnamese Phin": 450,
    "Cold Drip": 75, "Cold Brew": 560,
}

# Antioxidant rank (1-5) - from research papers (higher = more antioxidants)
ANTIOXIDANT_RANK = {
    "Ristretto": 4, "Espresso": 3, "Lungo": 5, "AeroPress": 1,
    "Moka Pot": 3, "Batch Brew": 2, "Pour Over": 2, "Siphon": 2,
    "French Press": 3, "Turkish": 3, "Vietnamese Phin": 3,
    "Cold Drip": 5, "Cold Brew": 2,
}

# Sensory scores (1-10 scale) - from specialty coffee associations
ACIDITY = {
    "Ristretto": 5, "Espresso": 6, "Lungo": 3, "AeroPress": 5, "Moka Pot": 4,
    "Pour Over": 8, "Batch Brew": 5, "Siphon": 7, "French Press": 4,
    "Turkish": 3, "Vietnamese Phin": 3, "Cold Drip": 2, "Cold Brew": 2,
}

BITTERNESS = {
    "Ristretto": 8, "Espresso": 7, "Lungo": 8, "AeroPress": 4, "Moka Pot": 6,
    "Pour Over": 3, "Batch Brew": 4, "Siphon": 3, "French Press": 6,
    "Turkish": 8, "Vietnamese Phin": 6, "Cold Drip": 2, "Cold Brew": 2,
}

BODY = {
    "Ristretto": 9, "Espresso": 9, "Lungo": 6, "AeroPress": 5, "Moka Pot": 8,
    "Pour Over": 4, "Batch Brew": 5, "Siphon": 5, "French Press": 8,
    "Turkish": 8, "Vietnamese Phin": 7, "Cold Drip": 5, "Cold Brew": 6,
}

# Complexity rating
COMPLEXITY = {
    "Ristretto": "High", "Espresso": "High", "Lungo": "High", "AeroPress": "Low",
    "Moka Pot": "Medium", "Pour Over": "High", "Batch Brew": "Low", "Siphon": "High",
    "French Press": "Low", "Turkish": "High", "Vietnamese Phin": "Low",
    "Cold Drip": "Low", "Cold Brew": "Low",
}

# Equipment needed
EQUIPMENT = {
    "Ristretto": "Espresso Machine", "Espresso": "Espresso Machine", "Lungo": "Espresso Machine",
    "AeroPress": "AeroPress", "Moka Pot": "Moka Pot",
    "Pour Over": "Pour Over Dripper + Gooseneck Kettle", "Batch Brew": "Automatic Dripper",
    "Siphon": "Siphon Brewer", "French Press": "French Press", "Turkish": "Ibrik (Cezve)",
    "Vietnamese Phin": "Vietnamese Phin", "Cold Drip": "Cold Drip Tower",
    "Cold Brew": "Glass Jar + Mesh Filter",
}

# Quick brewing guide
BREW_GUIDE = {
    "Ristretto": "Grind extra-fine → extract 20-25 sec → collect only first 25ml",
    "Espresso": "Grind fine → tamp 15kg → extract 25-30 sec → 36ml yield",
    "Lungo": "Grind fine → extract longer 35-45 sec → 110ml yield",
    "AeroPress": "Medium grind → water 85°C → steep 1-2 min → press gently 20-30 sec",
    "Moka Pot": "Medium-fine grind → medium heat → remove when hissing sound begins",
    "Pour Over": "Medium grind → water 95°C → pour in 5 stages → total 3.5-4 min",
    "Batch Brew": "Medium grind → 1:16 ratio → brew 4-5 min → serve immediately",
    "Siphon": "Heat water to rise → stir 50 sec → remove heat to draw down",
    "French Press": "Coarse grind → water 93°C → steep 4 min → press slowly 20 sec",
    "Turkish": "Powder-fine grind → heat 2-3 times until foaming → do not stir",
    "Vietnamese Phin": "Medium-coarse grind → bloom 30 sec → add water → wait 4-6 min",
    "Cold Drip": "Medium-coarse grind → 1 drop/sec → wait 8-12 hrs → dilute with ice",
    "Cold Brew": "Coarse grind → 1:8 ratio → refrigerate 12-24 hrs → filter grounds",
}

# Primary taste profile
PRIMARY_TASTE = {
    "Ristretto": "Sweet, Caramel, Fruity", "Espresso": "Balanced, Chocolate, Nutty",
    "Lungo": "Bitter, Woody, Spicy", "AeroPress": "Clean, Balanced, Sweet",
    "Moka Pot": "Rich, Chocolatey", "Pour Over": "Bright, Clean, Fruity, Floral",
    "Batch Brew": "Smooth, Balanced", "Siphon": "Aromatic, Complex, Floral",
    "French Press": "Rich, Full-bodied, Bold", "Turkish": "Strong, Thick, Spicy, Earthy",
    "Vietnamese Phin": "Bold, Chocolatey, Sweet", "Cold Drip": "Clean, Smooth, Light, Tea-like",
    "Cold Brew": "Smooth, Sweet, Low-acid, Chocolatey",
}

# Build master dataframe
master_rows = []

for method in ALL_METHODS:
    row = {
        "Method": method,
        "Brew Time (min)": BREW_TIME[method],
        "Caffeine (mg)": CAFFEINE_MG[method],
        "Antioxidant Rank (1-5)": ANTIOXIDANT_RANK[method],
        "Acidity (1-10)": ACIDITY[method],
        "Bitterness (1-10)": BITTERNESS[method],
        "Body (1-10)": BODY[method],
        "Complexity": COMPLEXITY[method],
        "Recommended Equipment": EQUIPMENT[method],
        "Quick Brewing Guide": BREW_GUIDE[method],
        "Primary Taste": PRIMARY_TASTE[method],
    }
    master_rows.append(row)

master_df = pd.DataFrame(master_rows)
master_df = master_df.sort_values("Brew Time (min)").reset_index(drop=True)

print(f" Master dataset built: {len(master_df)} rows × {len(master_df.columns)} columns")
print("\n Master data preview:")
print(master_df.head(13).to_string())

# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("DATA VALIDATION")
print("=" * 60)

print("\n Missing values check:")
missing = master_df.isnull().sum()
print(missing[missing > 0] if missing.any() else " No missing values")

print("\n Numeric ranges validation:")
numeric_cols = ["Brew Time (min)", "Caffeine (mg)", "Antioxidant Rank (1-5)",
               "Acidity (1-10)", "Bitterness (1-10)", "Body (1-10)"]
for col in numeric_cols:
    min_val = master_df[col].min()
    max_val = master_df[col].max()
    print(f"   {col}: {min_val} → {max_val}")

print(f"\n Complexity values: {sorted(master_df['Complexity'].unique())}")

# ============================================================
# EXPORT TO EXCEL (exactly like your original file)
# ============================================================

print("\n" + "=" * 60)
print("EXPORTING TO EXCEL")
print("=" * 60)

OUTPUT_FILE = "Coffee_Brewing_Methods_Dataset.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    # Sheet 1: Main dataset with title rows (matching your format)
    # We'll write with startrow=2 to leave space for title
    master_df.to_excel(writer, sheet_name="Coffee Brewing Methods", index=False, startrow=2)

    # Add summary statistics at bottom of sheet 1 (like your original)
    wb = writer.book
    ws = wb["Coffee Brewing Methods"]

    # Add title at top
    ws.merge_cells("A1:K1")
    ws["A1"] = "☕  COFFEE BREWING METHODS — COMPLETE GUIDE"
    ws["A1"].font = openpyxl.styles.Font(bold=True, size=14)

    ws.merge_cells("A2:K2")
    ws["A2"] = f"{len(master_df)} methods ranked by brew time · Data for educational & analytical use"
    ws["A2"].font = openpyxl.styles.Font(italic=True, size=10)

    # Add summary statistics rows (like your original file)
    start_row = len(master_df) + 5
    ws.cell(row=start_row, column=1, value="  SUMMARY STATISTICS")
    ws.cell(row=start_row, column=1).font = openpyxl.styles.Font(bold=True, size=12)

    summary_data = [
        ("Fastest Brew", master_df.loc[master_df["Brew Time (min)"].idxmin(), "Method"]),
        ("Slowest Brew", master_df.loc[master_df["Brew Time (min)"].idxmax(), "Method"]),
        ("Highest Caffeine", master_df.loc[master_df["Caffeine (mg)"].idxmax(), "Method"]),
        ("Avg Caffeine (mg)", round(master_df["Caffeine (mg)"].mean(), 0)),
        ("Avg Acidity", round(master_df["Acidity (1-10)"].mean(), 1)),
        ("Avg Bitterness", round(master_df["Bitterness (1-10)"].mean(), 1)),
        ("Avg Body", round(master_df["Body (1-10)"].mean(), 1)),
    ]

    for i, (label, value) in enumerate(summary_data):
        ws.cell(row=start_row + 1 + i, column=1, value=label)
        ws.cell(row=start_row + 1 + i, column=2, value=value)

    # Sheet 2: Raw scraped data (for transparency)
    raw_summary = []
    for item in wiki_raw_data:
        raw_summary.append({
            "source": item.get("source"),
            "method": item.get("method"),
            "description_snippet": item.get("description", "")[:200],
            "brew_time_scraped": item.get("brew_time_scraped", ""),
            "caffeine_scraped_mg": item.get("caffeine_scraped_mg", ""),
        })
    pd.DataFrame(raw_summary).to_excel(writer, sheet_name="Raw Scraped Data", index=False)

    # Sheet 3: Healthline caffeine data
    if healthline_raw_data:
        pd.DataFrame(healthline_raw_data).to_excel(writer, sheet_name="Healthline Caffeine Data", index=False)

print(f"\n Exported to: {OUTPUT_FILE}")
print(f"   - Sheet 1: Coffee Brewing Methods ({len(master_df)} rows)")
print(f"   - Sheet 2: Raw Scraped Data ({len(wiki_raw_data)} rows)")
print(f"   - Sheet 3: Healthline Caffeine Data ({len(healthline_raw_data)} rows)")

# ============================================================
# FINAL PREVIEW
# ============================================================

print("\n" + "=" * 60)
print("FINAL MASTER DATASET PREVIEW")
print("=" * 60)

print("\n Fastest Brew:")
print(master_df.nsmallest(3, "Brew Time (min)")[["Method", "Brew Time (min)", "Caffeine (mg)"]].to_string(index=False))

print("\n Highest Caffeine (Top 3):")
print(master_df.nlargest(3, "Caffeine (mg)")[["Method", "Caffeine (mg)", "Brew Time (min)"]].to_string(index=False))

print("\n Most Acidic (Top 3):")
print(master_df.nlargest(3, "Acidity (1-10)")[["Method", "Acidity (1-10)", "Primary Taste"]].to_string(index=False))

print("\n Fullest Body (Top 3):")
print(master_df.nlargest(3, "Body (1-10)")[["Method", "Body (1-10)", "Bitterness (1-10)"]].to_string(index=False))

print("\n" + "=" * 60)
print(" FULL PIPELINE COMPLETE!")
print("=" * 60)
print(f"   File name: {OUTPUT_FILE}")
print("\n Pipeline summary:")
print("   1. Scraped from: Wikipedia, Healthline, Perfect Daily Grind, NCA")
print("   2. Raw data saved to sheet 'Raw Scraped Data'")
print("   3. Cleaned & validated")
print("   4. Master dataset matches your original format")

☕ COFFEE BREWING METHODS — FULL SCRAPING PIPELINE

 Setup complete. Starting scraping...

SOURCE 1: Wikipedia - Coffee brewing methods
🔍 Scraping: Espresso...
    Scraped: Espresso - description length: 420
🔍 Scraping: Ristretto...
    Scraped: Ristretto - description length: 500
🔍 Scraping: Lungo...
    Scraped: Lungo - description length: 252
🔍 Scraping: AeroPress...
    Scraped: AeroPress - description length: 361
🔍 Scraping: Moka Pot...
    Scraped: Moka Pot - description length: 468
🔍 Scraping: Pour Over...
    Scraped: Pour Over - description length: 500
🔍 Scraping: French Press...
    Scraped: French Press - description length: 205
🔍 Scraping: Turkish...
    Scraped: Turkish - description length: 233
🔍 Scraping: Cold Brew...
    Scraped: Cold Brew - description length: 256
🔍 Scraping: Siphon...
    Scraped: Siphon - description length: 233
🔍 Scraping: Vietnamese Phin...
    Scraped: Vietnamese Phin - description length: 450
🔍 Scraping: Cold Drip...
    Scraped: Cold Drip - descr